# Multiple Linear Regression from Scratch
# Student Score Prediction

---

**Repository:** Machine Learning from Scratch  
**Notebook:** 03 — Multiple Linear Regression with Manual Normalization  
**Algorithm:** Multiple Linear Regression via Gradient Descent (no Scikit-Learn)  
**Scaling:** Manual Min-Max Normalization (no StandardScaler)  
**Author:** Ather-ops  

---

## Objective

Predict a student's **final exam score** from three behavioral features — study hours, sleep hours, and attendance — using Multiple Linear Regression implemented completely from scratch with NumPy.

---

## What is New in This Notebook vs Notebook 02

| Aspect | Notebook 02 | This Notebook |
|--------|------------|---------------|
| Features | 1 (Hours) | 3 (Study, Sleep, Attendance) |
| Dataset | Inline dict | CSV file |
| Scaling | None — small values | Manual normalization — divide by max |
| Gradient sign | `m -= alpha * gradient` | Same, but `error = Y - y_pred` (sign flipped) |
| Epochs | 50 | 2000 |
| Learning rate | 0.01 | 0.00001 |

---

## The Model

$$\hat{score} = m_1 \cdot study\_hours_{norm} + m_2 \cdot sleep\_hours_{norm} + m_3 \cdot attendance_{norm} + b$$

All three features are normalized before training. The weights $m_1, m_2, m_3$ are learned by gradient descent from zero.

---

## Manual Normalization — Divide by Max

$$x_{norm} = \frac{x}{x_{max}}$$

This scales every feature to the range [0, 1]. It is simpler than StandardScaler (which uses mean and std) but achieves the same core goal: putting all features on a comparable numerical scale so gradient descent converges smoothly.

**Why normalization is essential here:**  
Attendance ranges from 0 to 100. Study hours ranges from 0 to 24. Without normalization, attendance raw values are 4x larger than study hours — the gradient for attendance would dominate and the parameters for study hours would update too slowly.

---

## Gradient Sign Convention in This Notebook

In Notebook 01 and 02, error was defined as `y_pred - Y` (prediction minus actual).  
In this notebook, error is defined as `Y - y_pred` (actual minus prediction).

This flips the sign of the gradients:

| Convention | Error | Gradient formula | Update rule |
|------------|-------|-----------------|-------------|
| Notebooks 01-02 | `y_pred - Y` | `+(2/n) * sum(error * X)` | `m -= lr * gradient` |
| This notebook | `Y - y_pred` | `-(2/n) * sum(X * error)` | `m -= lr * dm` |

Both are mathematically equivalent — the negative sign in `dm` compensates for the flipped error definition. The parameters update in exactly the same direction.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset and inspect |
| 3 | Exploratory Data Analysis |
| 4 | Extract features and target |
| 5 | Manual normalization |
| 6 | Initialize parameters |
| 7 | Gradient descent training loop |
| 8 | Final parameters |
| 9 | Loss curve visualization |
| 10 | Post-training visualization |
| 11 | Predict score for a new student |
| 12 | New student prediction visualization |

---
## Step 1 — Import Libraries

Same three libraries as every notebook in this repo. No Scikit-Learn anywhere.

| Library | Role in this notebook |
|---------|----------------------|
| `numpy` | All gradient math — vectorized sum operations on arrays |
| `pandas` | Load CSV, inspect DataFrame, extract feature arrays |
| `matplotlib.pyplot` | Loss curve and prediction visualizations |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Step 2 — Load Dataset

We load the student scores CSV and immediately print the first few rows and basic info.

**Expected columns:**

| Column | Type | Description |
|--------|------|-------------|
| `study_hours` | float | Daily hours spent studying |
| `sleep_hours` | float | Daily hours of sleep |
| `attendance` | float | Class attendance percentage (0-100) |
| `final_score` | float | Final exam score — target variable |

**Note on the file extension:**  
The CSV is loaded as `student_scores_csv` without a `.csv` extension. If your file has the extension, update the filename to `student_scores_csv.txt` or `student_scores.csv` accordingly.

In [ ]:
# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("student_scores_csv")
print("Original Data:")
print(df.head())
print("=" * 80)
print("Shape          :", df.shape)
print("Columns        :", df.columns.tolist())
print("Missing values :", df.isnull().sum().sum())
print("=" * 80)

---
## Step 3 — Exploratory Data Analysis

Before extracting arrays and normalizing, we visualize the raw relationships between each feature and the target.

**What to look for:**

| Feature | Expected trend | If flat |
|---------|---------------|--------|
| `study_hours` | Upward — more study, higher score | Weak predictor |
| `sleep_hours` | Mild upward — rest aids performance | Noisy but slightly positive |
| `attendance` | Upward — more exposure, better score | Low attendance dataset variation |

We also print descriptive statistics and check the max value of each feature — these max values are used in the normalization step.

In [ ]:
# EDA — statistics and scatter plots
print("Descriptive Statistics:")
print("=" * 80)
print(df.describe().round(3))
print("=" * 80)
print("Max values used for normalization:")
print(f"  study_hours max  : {df['study_hours'].max()}")
print(f"  sleep_hours max  : {df['sleep_hours'].max()}")
print(f"  attendance max   : {df['attendance'].max()}")
print("=" * 80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df["study_hours"], df["final_score"],
                color="steelblue", edgecolors="white", s=60, alpha=0.8)
axes[0].set_xlabel("Study Hours")
axes[0].set_ylabel("Final Score")
axes[0].set_title("Study Hours vs Final Score")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].scatter(df["sleep_hours"], df["final_score"],
                color="darkorange", edgecolors="white", s=60, alpha=0.8)
axes[1].set_xlabel("Sleep Hours")
axes[1].set_ylabel("Final Score")
axes[1].set_title("Sleep Hours vs Final Score")
axes[1].grid(True, linestyle="--", alpha=0.5)

axes[2].scatter(df["attendance"], df["final_score"],
                color="seagreen", edgecolors="white", s=60, alpha=0.8)
axes[2].set_xlabel("Attendance (%)")
axes[2].set_ylabel("Final Score")
axes[2].set_title("Attendance vs Final Score")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("EDA — Raw Feature Relationships (Before Training)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("=" * 80)

---
## Step 4 — Extract Features and Target

We extract each column as a NumPy array. Working with individual 1D arrays keeps every gradient formula readable and directly maps to the math.

| Variable | Source | Shape | Role |
|----------|--------|-------|------|
| `X1` | study_hours | (n,) | Feature 1 |
| `X2` | sleep_hours | (n,) | Feature 2 |
| `X3` | attendance | (n,) | Feature 3 |
| `Y` | final_score | (n,) | Target |

The max values of each feature are stored **before** normalization. They are needed again in Step 11 to normalize the new student's input using the same scaling that was applied to the training data.

In [ ]:
# -----------------------------
# Features & Target
# -----------------------------
X1 = df["study_hours"].values
X2 = df["sleep_hours"].values
X3 = df["attendance"].values
Y  = df["final_score"].values

# Store max values before normalization — needed for new predictions
max_study      = df["study_hours"].max()
max_sleep      = df["sleep_hours"].max()
max_attendance = df["attendance"].max()

print("Feature Arrays Extracted")
print("=" * 80)
print(f"X1 (study_hours) shape : {X1.shape}  range: [{X1.min():.1f}, {X1.max():.1f}]")
print(f"X2 (sleep_hours) shape : {X2.shape}  range: [{X2.min():.1f}, {X2.max():.1f}]")
print(f"X3 (attendance)  shape : {X3.shape}  range: [{X3.min():.1f}, {X3.max():.1f}]")
print(f"Y  (final_score) shape : {Y.shape}   range: [{Y.min():.1f}, {Y.max():.1f}]")
print("=" * 80)

---
## Step 5 — Manual Feature Normalization

We normalize each feature by dividing it by its maximum value:

$$x_{norm} = \frac{x}{x_{max}}$$

After normalization, all three features lie in the range [0, 1].

**Why this matters for gradient descent:**

Without normalization, the gradient for `attendance` is computed on values up to 100, while `study_hours` values are up to ~14. The gradient magnitudes are therefore very different:

$$\frac{\partial L}{\partial m_3} = \frac{2}{n} \sum error \times attendance$$

Attendance raw values of ~80-100 make this gradient large. Study hours values of ~8-12 make the corresponding gradient small. After normalization, all three gradients operate on values between 0 and 1, so parameter updates are balanced.

**This notebook uses divide-by-max rather than StandardScaler:**  
The Scikit-Learn repo uses StandardScaler (mean=0, std=1). Both methods normalize — this version is simpler, fully transparent, and illustrates the core concept without any library dependency.

In [ ]:
# -----------------------------
# Feature Scaling (Normalization)
# -----------------------------
X1 = X1 / X1.max()
X2 = X2 / X2.max()
X3 = X3 / X3.max()

print("After Normalization (divide by max):")
print("=" * 80)
print(f"X1 (study_hours) range: [{X1.min():.4f}, {X1.max():.4f}]")
print(f"X2 (sleep_hours) range: [{X2.min():.4f}, {X2.max():.4f}]")
print(f"X3 (attendance)  range: [{X3.min():.4f}, {X3.max():.4f}]")
print("All features now in [0, 1] — gradient descent will converge stably.")
print("=" * 80)

---
## Step 6 — Initialize Parameters

All four learnable parameters start at zero. The training loop will update them through 2000 gradient descent steps.

| Parameter | Initial value | What it learns |
|-----------|--------------|----------------|
| `m1` | 0.0 | Contribution of normalized study hours to score |
| `m2` | 0.0 | Contribution of normalized sleep hours to score |
| `m3` | 0.0 | Contribution of normalized attendance to score |
| `b` | 0.0 | Base score when all normalized features are zero |

**Learning rate `lr = 0.00001`:**  
Even after normalization, the features are in [0, 1] but the target `final_score` can be up to 100. The product `error * X` in the gradient involves final score residuals (potentially large) multiplied by normalized features (small). A small learning rate prevents overshooting.

**2000 epochs:**  
More than Notebook 02 (50 steps) because:
1. Three features means a more complex loss surface
2. The learning rate is smaller so each step moves less
3. Printing only every 200 epochs keeps output clean

In [ ]:
# -----------------------------
# Initialize Parameters
# -----------------------------
m1 = m2 = m3 = b = 0.0
lr           = 0.00001
epochs       = 2000
n            = len(Y)
loss_history = []

print("Parameters Initialized")
print("=" * 80)
print(f"m1 = {m1}, m2 = {m2}, m3 = {m3}, b = {b}")
print(f"Learning rate (lr) : {lr}")
print(f"Epochs             : {epochs}")
print(f"Training samples   : {n}")
print(f"Total updates      : {epochs} (one full-batch update per epoch)")
print("=" * 80)

---
## Step 7 — Gradient Descent Training Loop

The training loop runs for 2000 epochs. At every epoch:

```
1. Forward pass      : y_pred = m1*X1 + m2*X2 + m3*X3 + b
2. Compute error     : error = Y - y_pred    (actual minus predicted)
3. Compute MSE loss  : loss = mean(error^2)
4. Compute gradients : dm1, dm2, dm3, db
5. Update parameters : m1 -= lr * dm1  etc.
6. Store loss        : loss_history.append(loss)
```

**Gradient formulas in this notebook:**

$$dm_1 = -\frac{2}{n} \sum X_1 \cdot error$$

$$dm_2 = -\frac{2}{n} \sum X_2 \cdot error$$

$$dm_3 = -\frac{2}{n} \sum X_3 \cdot error$$

$$db = -\frac{2}{n} \sum error$$

The negative sign in the gradient formulas is because `error = Y - y_pred`. When the model underestimates (`y_pred < Y`), error is positive, the gradient is negative, so `m -= lr * (negative)` increases `m` — which is the correct direction to reduce underestimation.

**Print every 200 epochs** to monitor training without flooding the output.

In [ ]:
# -----------------------------
# Training Loop (Gradient Descent)
# -----------------------------
for epoch in range(epochs):
    y_pred = m1*X1 + m2*X2 + m3*X3 + b
    error  = Y - y_pred
    loss   = np.mean(error ** 2)
    loss_history.append(loss)
    dm1 = -(2/n) * np.sum(X1 * error)
    dm2 = -(2/n) * np.sum(X2 * error)
    dm3 = -(2/n) * np.sum(X3 * error)
    db  = -(2/n) * np.sum(error)
    m1 -= lr * dm1
    m2 -= lr * dm2
    m3 -= lr * dm3
    b  -= lr * db
    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Loss: {loss:.4f}")

---
## Step 8 — Final Parameters

After 2000 epochs, the model has converged to its best-fit parameters. We print the final values and interpret their meaning.

**Reading the learned weights:**

Because all features were normalized to [0, 1], the weights are directly comparable:
- The feature with the **largest positive weight** contributes most to predicting a high score
- A **negative weight** (unexpected here) would mean that feature is inversely associated with score in this dataset

Expected: `m1 (study)` should be the largest positive weight — study hours is the strongest behavioral predictor of exam scores.

In [ ]:
# -----------------------------
# Final Parameters
# -----------------------------
print("=" * 80)
print("Trained Model Parameters:")
print(f"m1 (study): {m1:.4f}")
print(f"m2 (sleep): {m2:.4f}")
print(f"m3 (attendance): {m3:.4f}")
print(f"b (bias): {b:.4f}")
print("=" * 80)

# Feature importance from weights
weights = {"study_hours": abs(m1), "sleep_hours": abs(m2), "attendance": abs(m3)}
ranked  = sorted(weights.items(), key=lambda x: x[1], reverse=True)
print("Feature influence ranking (by weight magnitude):")
for rank, (feat, w) in enumerate(ranked, 1):
    print(f"  {rank}. {feat:15s}: |weight| = {w:.4f}")
print("=" * 80)

# Final loss
print(f"Final training loss (MSE) : {loss_history[-1]:.4f}")
print(f"Final RMSE                : {np.sqrt(loss_history[-1]):.4f}")
print("=" * 80)

---
## Step 9 — Loss Curve Visualization

We plot the MSE loss across all 2000 epochs. With 2000 epochs and a small learning rate, the curve should show a longer gradual descent compared to Notebook 02's steep 50-step drop.

**What to look for:**

| Pattern | Meaning |
|---------|--------|
| Smooth continuous decrease | Healthy training — learning rate is well chosen |
| Sharp drop then plateau | Good — fast initial learning, then diminishing returns near optimum |
| Still clearly decreasing at epoch 2000 | More epochs needed, or learning rate could be increased |
| Flat from the start | Learning rate too small — try 10x larger |
| Spikes upward | Learning rate too large — try 10x smaller |

We also zoom into the first 200 epochs to see the steepest part of the descent.

In [ ]:
# -----------------------------
# Loss Visualization
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full loss curve
axes[0].plot(loss_history, color="green", linewidth=1.5)
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss Curve — All 2000 Epochs")
axes[0].grid(True)

# Early phase zoom
axes[1].plot(loss_history[:200], color="steelblue", linewidth=1.5)
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("MSE Loss")
axes[1].set_title("Training Loss Curve — First 200 Epochs")
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("=" * 80)
print(f"Initial loss (epoch 0)    : {loss_history[0]:.4f}")
print(f"Loss at epoch 200         : {loss_history[200]:.4f}")
print(f"Loss at epoch 1000        : {loss_history[1000]:.4f}")
print(f"Final loss  (epoch 1999)  : {loss_history[-1]:.4f}")
pct = (1 - loss_history[-1] / loss_history[0]) * 100
print(f"Total loss reduction      : {pct:.2f}%")
print("=" * 80)

---
## Step 10 — Post-Training Visualization

We generate final predictions on the full training set and visualize how well the model fits the data — one plot per feature, plus an Actual vs Predicted diagnostic scatter.

**What to look for:**
- Predicted dots (green) clustering close to actual dots (red) indicates a good fit
- In the Actual vs Predicted plot, tight clustering around the diagonal line = high accuracy

In [ ]:
# Post-training predictions and visualization
y_pred_final = m1*X1 + m2*X2 + m3*X3 + b

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Panel 1: Study hours vs score
axes[0].scatter(X1, Y,           color="red",      alpha=0.5, s=50, label="Actual")
axes[0].scatter(X1, y_pred_final, color="green",    alpha=0.5, s=50, label="Predicted")
axes[0].set_xlabel("Study Hours (normalized)")
axes[0].set_ylabel("Final Score")
axes[0].set_title("Study Hours vs Score")
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Panel 2: Sleep hours vs score
axes[1].scatter(X2, Y,           color="red",      alpha=0.5, s=50, label="Actual")
axes[1].scatter(X2, y_pred_final, color="blue",     alpha=0.5, s=50, label="Predicted")
axes[1].set_xlabel("Sleep Hours (normalized)")
axes[1].set_ylabel("Final Score")
axes[1].set_title("Sleep Hours vs Score")
axes[1].legend(fontsize=8)
axes[1].grid(True, linestyle="--", alpha=0.5)

# Panel 3: Attendance vs score
axes[2].scatter(X3, Y,           color="red",      alpha=0.5, s=50, label="Actual")
axes[2].scatter(X3, y_pred_final, color="orange",   alpha=0.5, s=50, label="Predicted")
axes[2].set_xlabel("Attendance (normalized)")
axes[2].set_ylabel("Final Score")
axes[2].set_title("Attendance vs Score")
axes[2].legend(fontsize=8)
axes[2].grid(True, linestyle="--", alpha=0.5)

# Panel 4: Actual vs Predicted
axes[3].scatter(Y, y_pred_final, color="steelblue",
                edgecolors="white", s=70, alpha=0.8)
min_val = min(Y.min(), y_pred_final.min()) - 3
max_val = max(Y.max(), y_pred_final.max()) + 3
axes[3].plot([min_val, max_val], [min_val, max_val],
             color="red", linestyle="--", linewidth=1.5,
             label="Perfect prediction")
axes[3].set_xlabel("Actual Score")
axes[3].set_ylabel("Predicted Score")
axes[3].set_title("Actual vs Predicted")
axes[3].legend(fontsize=8)
axes[3].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Post-Training Visualization — After 2000 Epochs",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("=" * 80)

---
## Step 11 — Predict Score for a New Student

We predict the final score for a student with:

| Feature | Raw Value | Context |
|---------|----------|--------|
| study | 7 hours/day | Above average |
| sleep | 8 hours/day | Good — recommended amount |
| attendance | 85% | Decent — misses ~1 in 7 classes |

**Critical step — apply the same normalization:**  
The model was trained on normalized features. The new input must be normalized using the **same max values** from the training data — not the max of the new input alone.

This is why we stored `max_study`, `max_sleep`, and `max_attendance` in Step 4, **before** normalizing the training arrays. Using the training-set max values ensures the new input is on the same scale the model learned from.

If we forgot to normalize: the model would receive a raw value of 85 for attendance but was trained on values of 0.0 to 1.0 — the prediction would be wildly incorrect.

In [ ]:
# -----------------------------
# Prediction (New Student)
# -----------------------------
study      = 7
sleep      = 8
attendance = 85
study      /= df["study_hours"].max()
sleep      /= df["sleep_hours"].max()
attendance /= df["attendance"].max()
predicted_score = m1*study + m2*sleep + m3*attendance + b
print("=" * 80)
print(f"Predicted Final Score: {predicted_score:.2f}")
print("=" * 80)

# Step by step breakdown
print("Calculation breakdown:")
print(f"  study      normalized : {study:.4f}  (7 / {df['study_hours'].max()})")
print(f"  sleep      normalized : {sleep:.4f}  (8 / {df['sleep_hours'].max()})")
print(f"  attendance normalized : {attendance:.4f}  (85 / {df['attendance'].max()})")
print(f"  m1 * study            : {m1 * study:.4f}")
print(f"  m2 * sleep            : {m2 * sleep:.4f}")
print(f"  m3 * attendance       : {m3 * attendance:.4f}")
print(f"  b                     : {b:.4f}")
print(f"  Total (predicted)     : {predicted_score:.4f}")
print("=" * 80)

---
## Step 12 — New Student Prediction Visualization

We place the new student's predicted score in context — overlaid on the training data distribution across all three features. A red star marks exactly where the new student sits relative to the full dataset.

This is the clearest way to answer: does this prediction look reasonable?

In [ ]:
# New student prediction visualization
raw_study      = 7
raw_sleep      = 8
raw_attendance = 85

# Normalized values for plotting
norm_study      = raw_study      / max_study
norm_sleep      = raw_sleep      / max_sleep
norm_attendance = raw_attendance / max_attendance

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (X1, "Study Hours (normalized)",  norm_study),
    (X2, "Sleep Hours (normalized)",  norm_sleep),
    (X3, "Attendance (normalized)",   norm_attendance),
]

for i, (feat_arr, xlabel, new_val) in enumerate(configs):
    axes[i].scatter(feat_arr, Y,
                    color="steelblue", alpha=0.5, s=50,
                    label="Training data", zorder=2)
    axes[i].scatter(new_val, predicted_score,
                    color="red", marker="*", s=350, zorder=5,
                    label=f"New student: {predicted_score:.1f}")
    axes[i].vlines(new_val, Y.min() - 2, predicted_score,
                   colors="gray", linestyles="dotted", linewidth=1.2)
    axes[i].hlines(predicted_score, feat_arr.min(), new_val,
                   colors="gray", linestyles="dotted", linewidth=1.2)
    axes[i].set_xlabel(xlabel)
    axes[i].set_ylabel("Final Score")
    axes[i].set_title(f"{xlabel.split(' (')[0]} — New Student")
    axes[i].legend(fontsize=9)
    axes[i].grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    f"New Student — Study: {raw_study}h  Sleep: {raw_sleep}h  "
    f"Attendance: {raw_attendance}%  |  Predicted Score: {predicted_score:.2f}",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()
print("=" * 80)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import numpy, pandas, matplotlib | No Scikit-Learn |
| 2 | Load CSV and inspect | Shape, columns, missing values |
| 3 | EDA scatter plots + max values | Visual relationships before training |
| 4 | Extract X1, X2, X3, Y as arrays | Store max values before normalization |
| 5 | Divide each feature by its max | All features scaled to [0, 1] |
| 6 | Initialize m1=m2=m3=b=0, lr=0.00001 | Starting from zero knowledge |
| 7 | 2000-epoch gradient descent loop | Forward pass, error, gradients, update |
| 8 | Print final m1, m2, m3, b + ranking | Feature influence by weight magnitude |
| 9 | Loss curve — full + first 200 zoom | Convergence confirmed visually |
| 10 | 4-panel post-training visualization | Actual vs predicted per feature + diagonal |
| 11 | New student prediction | Normalize with training max values first |
| 12 | New student star plot | Prediction in context of full dataset |

---

## Progression Across This Repo

| Notebook | Algorithm | Features | Normalization | Gradient Descent |
|----------|-----------|----------|--------------|------------------|
| 01 | Multiple Linear Regression | 3 (Area, Bedrooms, Age) | None | Mini-batch, 1000 epochs |
| 02 | Simple Linear Regression | 1 (Hours) | None | Normal Equation + 50-step full-batch |
| 03 (this) | Multiple Linear Regression | 3 (Study, Sleep, Attendance) | Divide by max | Full-batch, 2000 epochs |

Each notebook adds one new concept: Notebook 01 introduced batching, Notebook 02 introduced the closed-form solution, this notebook introduces manual normalization and shows the gradient sign convention explicitly.

---

**Next notebook:** `04_logistic_regression_scratch.ipynb` — Binary classification with sigmoid activation and binary cross-entropy, implemented from scratch